# 💳 Fraud Detection Model

## 🎯 Amaç
Bu notebook'un amacı, işlem verilerini kullanarak dolandırıcılık (fraud) tespiti yapan makine öğrenmesi modelleri geliştirmek ve karşılaştırmaktır.

## 📌 Yaklaşım
Bu çalışmada:
- Hazır hazırlanmış MVP veri seti yüklenecek
- Eğitim ve test verisine ayrılacak
- Veri dengesizliği SMOTE ile ele alınacak
- Logistic Regression, Random Forest ve XGBoost modelleri kurulacak
- Threshold tuning yapılacak
- En iyi model seçilecek ve kaydedilecek

## 📊 Değerlendirme Metrikleri
Fraud problemi dengesiz bir sınıflandırma problemidir. Bu nedenle şu metriklere odaklanacağız:

- Precision
- Recall
- F1 Score
- ROC AUC
- Average Precision

In [2]:
import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    precision_recall_curve
)

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

from imblearn.over_sampling import SMOTE

In [3]:
df = pd.read_parquet("../data/processed/transaction_features_mvp_sample.parquet")

print("Veri boyutu:", df.shape)
df.head()

Veri boyutu: (279972, 40)


,amount,log_amount,amount_zscore,high_amount,amount_to_limit_ratio,hour,day,month,year,day_of_week,...,user_mean_amount,user_std_amount,amount_deviation,time_diff,fast_tx,very_fast_tx,rolling_mean_3,rolling_std_3,rolling_amount_deviation,is_fraud
0,38.44,3.674781,-0.194635,0,0.001544,12,7,12,2018,Friday,...,62.180368,97.043952,-23.740368,2100.0,0,0,18.020000,18.723109,20.420000,0
1,4.71,1.742219,-0.641699,0,0.000293,11,22,7,2012,Sunday,...,29.201489,67.938937,-24.491489,18060.0,0,0,24.746667,32.335309,-20.036667,0
2,34.77,3.577110,-0.243278,0,0.001610,12,9,12,2017,Saturday,...,55.730116,59.643552,-20.960116,13020.0,0,0,48.306667,12.104596,-13.536667,0
3,57.00,4.060443,0.051363,0,0.002738,9,15,6,2014,Sunday,...,42.804189,47.763781,14.195811,540.0,0,0,42.226667,25.588164,14.773333,0
4,54.00,4.007333,0.011601,0,0.003971,7,27,7,2011,Wednesday,...,77.823015,104.888382,-23.823015,420.0,0,0,59.016667,8.689122,-5.016667,0


## 📊 Hedef Değişken Dağılımı

Bu bölümde fraud ve non-fraud oranını kontrol ediyoruz.
MVP veri seti yaklaşık %5 fraud içerecek şekilde hazırlanmıştır.

In [4]:
print(df["is_fraud"].value_counts())
print(df["is_fraud"].value_counts(normalize=True))

is_fraud
0    266640
1     13332
Name: count, dtype: int64
is_fraud
0    0.952381
1    0.047619
Name: proportion, dtype: float64


In [5]:
# Feature ve target ayırma
X = df.drop("is_fraud", axis=1)
y = df["is_fraud"]

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (279972, 39)
y shape: (279972,)


## 🛠️ Kategorik Değişkenlerin Dönüştürülmesi

SMOTE yalnızca sayısal veri ile çalıştığı için kategorik değişkenleri sayısal hale getiriyoruz.
Bu aşamada `pd.get_dummies()` kullanıyoruz.

In [6]:
X_encoded = pd.get_dummies(X, drop_first=True)

print("Encode sonrası boyut:", X_encoded.shape)
X_encoded.head()

Encode sonrası boyut: (279972, 6924)


,amount,log_amount,amount_zscore,high_amount,amount_to_limit_ratio,hour,day,month,year,day_of_week_num,...,mcc_8021,mcc_8041,mcc_8043,mcc_8049,mcc_8062,mcc_8099,mcc_8111,mcc_8931,mcc_9402,gender_Male
0,38.44,3.674781,-0.194635,0,0.001544,12,7,12,2018,4,...,False,False,False,False,False,False,False,False,False,True
1,4.71,1.742219,-0.641699,0,0.000293,11,22,7,2012,6,...,False,False,False,False,False,False,False,False,False,True
2,34.77,3.577110,-0.243278,0,0.001610,12,9,12,2017,5,...,False,False,False,False,False,False,False,False,False,False
3,57.00,4.060443,0.051363,0,0.002738,9,15,6,2014,6,...,False,False,False,False,False,False,False,False,False,True
4,54.00,4.007333,0.011601,0,0.003971,7,27,7,2011,2,...,False,False,False,False,False,False,False,False,False,True


## 🔀 Eğitim ve Test Verisinin Ayrılması

Veriyi eğitim ve test olarak ayırıyoruz.
Fraud oranının korunması için `stratify=y` kullanıyoruz.

In [7]:
X_train, X_test, y_train, y_test = train_test_split(
    X_encoded,
    y,
    test_size=0.20,
    stratify=y,
    random_state=42
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)
print("Train fraud oranı:", y_train.mean())
print("Test fraud oranı:", y_test.mean())

Train shape: (223977, 6924)
Test shape: (55995, 6924)
Train fraud oranı: 0.047620961080825266
Test fraud oranı: 0.047611393874453074


## ⚖️ Veri Dengesizliği ve SMOTE

Fraud sınıfı yine de minority olduğu için yalnızca eğitim verisi üzerinde SMOTE uyguluyoruz.

Önemli not:
- Test verisine kesinlikle SMOTE uygulanmaz
- Sadece eğitim verisi üzerinde kullanılır

In [8]:
print(y_train.value_counts())
print(y_train.value_counts(normalize=True))

is_fraud
0    213311
1     10666
Name: count, dtype: int64
is_fraud
0    0.952379
1    0.047621
Name: proportion, dtype: float64


In [9]:
train_df = X_train.copy()
train_df["is_fraud"] = y_train.values

In [10]:
train_fraud = train_df[train_df["is_fraud"] == 1].copy()
train_nonfraud = train_df[train_df["is_fraud"] == 0].copy()

print("Fraud:", len(train_fraud))
print("Non-fraud:", len(train_nonfraud))

Fraud: 10666
Non-fraud: 213311


In [11]:
desired_nonfraud = len(train_fraud) * 5

train_nonfraud_sample = train_nonfraud.sample(
    n=min(desired_nonfraud, len(train_nonfraud)),
    random_state=42
)

train_small = pd.concat([train_fraud, train_nonfraud_sample], axis=0)
train_small = train_small.sample(frac=1, random_state=42).reset_index(drop=True)

print(train_small["is_fraud"].value_counts())
print(train_small["is_fraud"].value_counts(normalize=True))

is_fraud
0    53330
1    10666
Name: count, dtype: int64
is_fraud
0    0.833333
1    0.166667
Name: proportion, dtype: float64


In [17]:
X_train_small = train_small.drop("is_fraud", axis=1)
y_train_small = train_small["is_fraud"]

from imblearn.over_sampling import SMOTE

smote = SMOTE(
    sampling_strategy=0.3,   # minority / majority = 0.3
    random_state=42
)

X_train_res, y_train_res = smote.fit_resample(X_train_small, y_train_small)

print(pd.Series(y_train_res).value_counts())
print(pd.Series(y_train_res).value_counts(normalize=True))

is_fraud
0    53330
1    15999
Name: count, dtype: int64
is_fraud
0    0.769231
1    0.230769
Name: proportion, dtype: float64


In [18]:
#VERİ BOYUTLRARINI KONTROL EDELİM;
#AMAÇ: train ve test setlerin beklediğin gibi mi görmek
    #SMOTE sonrası veri dengesi doğru mu kontrol etmek
print("X_train_res shape:", X_train_res.shape)
print("y_train_res shape:", y_train_res.shape)

print("X_test shape:", X_test.shape)
print("y_test shape:", y_test.shape)

print("\nSMOTE sonrası sınıf dağılımı:")
print(pd.Series(y_train_res).value_counts(normalize=True))

print("\nTest set sınıf dağılımı:")
print(pd.Series(y_test).value_counts(normalize=True))

X_train_res shape: (69329, 6924)
y_train_res shape: (69329,)
X_test shape: (55995, 6924)
y_test shape: (55995,)

SMOTE sonrası sınıf dağılımı:
is_fraud
0    0.769231
1    0.230769
Name: proportion, dtype: float64

Test set sınıf dağılımı:
is_fraud
0    0.952389
1    0.047611
Name: proportion, dtype: float64


In [20]:
#Model değerlendirme

from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score
)

def evaluate_model(model, X_test, y_test, threshold=0.5, model_name="Model"):
    y_proba = model.predict_proba(X_test)[:, 1]
    y_pred = (y_proba >= threshold).astype(int)

    print(f"\n=== {model_name} ===")
    print("\nConfusion Matrix:")
    print(confusion_matrix(y_test, y_pred))

    print("\nClassification Report:")
    print(classification_report(y_test, y_pred, zero_division=0))

    precision = precision_score(y_test, y_pred, zero_division=0)
    recall = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)
    roc_auc = roc_auc_score(y_test, y_proba)
    avg_precision = average_precision_score(y_test, y_proba)

    print("Precision:", precision)
    print("Recall:", recall)
    print("F1:", f1)
    print("ROC AUC:", roc_auc)
    print("Average Precision:", avg_precision)

    return {
        "model": model_name,
        "threshold": threshold,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "roc_auc": roc_auc,
        "avg_precision": avg_precision
    }

In [22]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_res_scaled = scaler.fit_transform(X_train_res)
X_test_scaled = scaler.transform(X_test)

In [23]:
from sklearn.linear_model import LogisticRegression

lr_model = LogisticRegression(
    max_iter=3000,      # artır
    solver="lbfgs",
    random_state=42
)

lr_model.fit(X_train_res_scaled, y_train_res)

LogisticRegression(max_iter=3000, random_state=42)

In [24]:
lr_result = evaluate_model(
    lr_model,
    X_test_scaled,
    y_test,
    model_name="Logistic Regression"
)


=== Logistic Regression ===

Confusion Matrix:
[[52463   866]
 [  270  2396]]

Classification Report:
              precision    recall  f1-score   support

           0       0.99      0.98      0.99     53329
           1       0.73      0.90      0.81      2666

    accuracy                           0.98     55995
   macro avg       0.86      0.94      0.90     55995
weighted avg       0.98      0.98      0.98     55995

Precision: 0.7345187001839363
Recall: 0.8987246811702926
F1: 0.8083670715249662
ROC AUC: 0.9812429832129412
Average Precision: 0.8771422888187057


In [25]:
#✔ Model güçlü
#✔ Overfitting yok (test performansı iyi)
#✔ Fraud yakalama çok başarılı
#✔ Dengeli precision/recall

In [26]:
#Threshold Tunning
from sklearn.metrics import precision_recall_curve
import numpy as np

y_proba = lr_model.predict_proba(X_test_scaled)[:, 1]

precision, recall, thresholds = precision_recall_curve(y_test, y_proba)

f1 = 2 * (precision * recall) / (precision + recall + 1e-9)

best_idx = np.argmax(f1[:-1])
best_threshold = thresholds[best_idx]

print("Best threshold:", best_threshold)

Best threshold: 0.7955836547747079


In [28]:
for t in [0.6]:
    print(f"\nThreshold: {t}")
    evaluate_model(lr_model, X_test_scaled, y_test, threshold=t)


Threshold: 0.6

=== Model ===

Confusion Matrix:
[[52685   644]
 [  314  2352]]

Classification Report:
              precision    recall  f1-score   support

           0       0.99      0.99      0.99     53329
           1       0.79      0.88      0.83      2666

    accuracy                           0.98     55995
   macro avg       0.89      0.94      0.91     55995
weighted avg       0.98      0.98      0.98     55995

Precision: 0.7850467289719626
Recall: 0.8822205551387847
F1: 0.8308018368067821
ROC AUC: 0.9812429832129412
Average Precision: 0.8771422888187057


In [29]:
#XGBoost Modeli
from xgboost import XGBClassifier

xgb_model = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    random_state=42,
    eval_metric="logloss",
    n_jobs=-1
)

In [30]:
#Modeli eğitirken XGBoost için scaling gerekmez
xgb_model.fit(X_train_res, y_train_res)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='logloss',
              feature_types=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=0.05, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=6,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=300,
              n_jobs=-1, num_parallel_tree=None, random_state=42, ...)

In [31]:
xgb_result = evaluate_model(
    xgb_model,
    X_test,
    y_test,
    threshold=0.6,
    model_name="XGBoost"
)


=== XGBoost ===

Confusion Matrix:
[[52913   416]
 [  210  2456]]

Classification Report:
              precision    recall  f1-score   support

           0       1.00      0.99      0.99     53329
           1       0.86      0.92      0.89      2666

    accuracy                           0.99     55995
   macro avg       0.93      0.96      0.94     55995
weighted avg       0.99      0.99      0.99     55995

Precision: 0.8551532033426184
Recall: 0.9212303075768942
F1: 0.8869628024557602
ROC AUC: 0.9965389020190975
Average Precision: 0.9494254485796142


In [33]:
lr_06_result = evaluate_model(
    lr_model,
    X_test_scaled,
    y_test,
    threshold=0.6,
    model_name="Logistic (0.6)"
)


=== Logistic (0.6) ===

Confusion Matrix:
[[52685   644]
 [  314  2352]]

Classification Report:
              precision    recall  f1-score   support

           0       0.99      0.99      0.99     53329
           1       0.79      0.88      0.83      2666

    accuracy                           0.98     55995
   macro avg       0.89      0.94      0.91     55995
weighted avg       0.98      0.98      0.98     55995

Precision: 0.7850467289719626
Recall: 0.8822205551387847
F1: 0.8308018368067821
ROC AUC: 0.9812429832129412
Average Precision: 0.8771422888187057


In [34]:
xgb_06_result = evaluate_model(
    xgb_model,
    X_test,
    y_test,
    threshold=0.6,
    model_name="XGBoost (0.6)"
)


=== XGBoost (0.6) ===

Confusion Matrix:
[[52913   416]
 [  210  2456]]

Classification Report:
              precision    recall  f1-score   support

           0       1.00      0.99      0.99     53329
           1       0.86      0.92      0.89      2666

    accuracy                           0.99     55995
   macro avg       0.93      0.96      0.94     55995
weighted avg       0.99      0.99      0.99     55995

Precision: 0.8551532033426184
Recall: 0.9212303075768942
F1: 0.8869628024557602
ROC AUC: 0.9965389020190975
Average Precision: 0.9494254485796142


In [35]:
import joblib

joblib.dump(xgb_model, "../models/fraud_model_final.joblib")


['../models/fraud_model_final.joblib']